# 07 — Measuring what representations add beyond basic information

The main benchmark shows how well each feature set predicts on its own. This notebook asks a different question: how much does a representation improve prediction after readily available location or property information is already included?

For PTAL, representations are added to the location baseline. For EPC, they are added separately to the compact property-control model and to the richer model containing floor area and construction age. Each comparison is made within the same held-out borough round, so the change in R², RMSE and MAE reflects the addition of features rather than a change in test locations.

The notebook also separates the contributions of EPC floor area and construction age. The five geographical rounds are used descriptively; they are not treated as five independent observations for a hypothesis test.

## Main findings

For PTAL, the location baseline has mean R² of 0.4015. Adding all representations and Street View availability raises it to 0.6931, an increase of 0.2917, with improvement in all five held-out rounds.

For EPC, the compact control model has mean R² of 0.1606. Adding the full representation set raises it to 0.3970, an increase of 0.2363 across all five rounds. DINOv2 accounts for much of this added value.

The richer EPC control model already reaches 0.5829. Adding TESSERA produces a small increase of 0.0058, while adding all representations lowers mean R² by 0.0115. The component analysis shows why this baseline is so strong: median floor area adds about 0.0028 R² to the compact controls, whereas construction age adds about 0.4223. The richer baseline is therefore dominated by direct information about building age.

In [ ]:
# Connect Google Drive and load packages for the controls-plus-representation analysis.
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import sys
import json
import gc
import time
import hashlib
import platform
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import sklearn

from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

FINAL_CODE_DIR = Path("/content/drive/MyDrive/GEOG0105/CODE/FINAL_PIPELINE")
if str(FINAL_CODE_DIR) in sys.path:
    sys.path.remove(str(FINAL_CODE_DIR))
sys.path.insert(0, str(FINAL_CODE_DIR))

import importlib
import config as _config
importlib.invalidate_caches()
_config = importlib.reload(_config)
globals().update({
    name: getattr(_config, name)
    for name in dir(_config)
    if not name.startswith("_")
})

pd.set_option("display.max_columns", 180)
pd.set_option("display.width", 240)

print("Python:", platform.python_version())
print("numpy:", np.__version__, "pandas:", pd.__version__, "sklearn:", sklearn.__version__)
print("Canonical table:", FINAL_MODEL_TABLE_PATH, FINAL_MODEL_TABLE_PATH.exists())
print("Feature manifest:", FEATURE_MANIFEST_JSON_PATH, FEATURE_MANIFEST_JSON_PATH.exists())
print("Frozen Notebook-06 folds:", RIDGE_OUTER_FOLDS_PATH, RIDGE_OUTER_FOLDS_PATH.exists())
print("Frozen Notebook-06 run spec:", RIDGE_RUN_SPEC_PATH, RIDGE_RUN_SPEC_PATH.exists())
print("Frozen Notebook-06 audit:", RIDGE_CORE_AUDIT_PATH, RIDGE_CORE_AUDIT_PATH.exists())

for required_path in [
    FINAL_MODEL_TABLE_PATH,
    FEATURE_MANIFEST_JSON_PATH,
    RIDGE_OUTER_FOLDS_PATH,
    RIDGE_RUN_SPEC_PATH,
    RIDGE_CORE_AUDIT_PATH,
]:
    assert required_path.exists(), f"Missing prerequisite: {required_path}"

## 1. Load the benchmark definitions and results

The modelling table, feature manifest, borough assignments and Ridge settings are read from the main benchmark. Their stored identifiers are compared so that the incremental models use the same samples, predictors and evaluation structure.

In [ ]:
# Read the established benchmark definitions, data identities and borough assignments.
df = pd.read_parquet(FINAL_MODEL_TABLE_PATH)
with open(FEATURE_MANIFEST_JSON_PATH, "r") as f:
    manifest = json.load(f)
with open(RIDGE_RUN_SPEC_PATH, "r") as f:
    source_06_spec = json.load(f)
with open(RIDGE_CORE_AUDIT_PATH, "r") as f:
    source_06_audit = json.load(f)

assert source_06_audit["integrity_gate_pass"] is True
assert source_06_audit["interpretation_gate_pass"] is True

target_col = manifest["target_column"]
group_col = manifest["group_column"]
categorical_master = set(manifest["categorical_columns"])
feature_sets = manifest["feature_sets"]

assert len(df) == 26597
assert df["sample_id"].is_unique
assert df[target_col].notna().all()
assert df[group_col].notna().all()
assert df[group_col].nunique() == 33
assert set(df["task"].unique()) == {"PTAL", "EPC"}

df = df.sort_values(["task", "sample_id"], kind="mergesort").reset_index(drop=True)
task_counts = df["task"].value_counts().to_dict()
assert task_counts == {"EPC": 20000, "PTAL": 6597}

model_key_frame = df[["sample_id", "task", group_col, target_col]].copy()
model_key_hash = hashlib.sha256(
    pd.util.hash_pandas_object(model_key_frame, index=False).values.tobytes()
).hexdigest()
manifest_hash = hashlib.sha256(
    json.dumps(manifest, sort_keys=True).encode("utf-8")
).hexdigest()

assert source_06_spec["model_key_sha256"] == model_key_hash
assert source_06_spec["feature_manifest_sha256"] == manifest_hash
assert int(source_06_spec["outer_splits"]) == int(RIDGE_OUTER_SPLITS) == 5
assert int(source_06_spec["inner_splits"]) == int(RIDGE_INNER_SPLITS) == 3
assert [float(x) for x in source_06_spec["alpha_grid"]] == [float(x) for x in RIDGE_ALPHA_GRID]

print("Notebook-06 integrity and interpretation gates: PASS")
print("Model-key SHA256:", model_key_hash)
print("Manifest SHA256:", manifest_hash)
display(pd.Series(task_counts, name="n"))

## 2. Define the controls-plus-representation models

Every representation set from the main benchmark is added to the relevant task baseline. Retaining the complete pre-defined list avoids selecting only the encoders that looked strongest in the first comparison.

The richer EPC baseline is labelled separately because floor area and construction age come from the same certificate records used to construct the EPC outcome. These are valid predictors, but they represent a more information-rich setting than can be assumed for a purely image-based application.

In [ ]:
# Construct the complete matrix of baselines, added representations and EPC components.
increment_rep_sets = [
    "SatCLIP",
    "TESSERA",
    "AlphaEarth",
    "DINOv2",
    "StreetView_CLIP_only",
    "StreetView_metadata_only",
    "StreetView_CLIP_plus_metadata",
    "Location_and_EO",
    "Sky_and_space",
    "Street_and_sky",
    "All_representations_without_SV_metadata",
    "All_representations_plus_SV_metadata",
]

representation_class = {
    "SatCLIP": "single_representation",
    "TESSERA": "single_representation",
    "AlphaEarth": "single_representation",
    "DINOv2": "single_representation",
    "StreetView_CLIP_only": "single_representation",
    "StreetView_metadata_only": "coverage_metadata",
    "StreetView_CLIP_plus_metadata": "single_representation_plus_metadata",
    "Location_and_EO": "theory_fusion",
    "Sky_and_space": "theory_fusion",
    "Street_and_sky": "theory_fusion",
    "All_representations_without_SV_metadata": "full_fusion",
    "All_representations_plus_SV_metadata": "full_fusion_plus_metadata",
}

required_base_sets = [
    "PTAL_spatial_baseline",
    "EPC_controls_sparse",
    "EPC_controls_extensive",
]
for name in required_base_sets + increment_rep_sets:
    assert name in feature_sets, f"Missing frozen feature set: {name}"
    assert feature_sets[name], f"Empty frozen feature set: {name}"
    assert len(feature_sets[name]) == len(set(feature_sets[name]))
    assert not [c for c in feature_sets[name] if c not in df.columns]

epc_sparse = list(feature_sets["EPC_controls_sparse"])
epc_extensive = list(feature_sets["EPC_controls_extensive"])
extensive_increment = [c for c in epc_extensive if c not in epc_sparse]
assert extensive_increment == [
    "median_total_floor_area",
    "construction_age_band_mode",
], extensive_increment

def dedupe_preserve_order(columns):
    return list(dict.fromkeys(columns))

model_records = []
model_features = {}

def add_model(task, model_id, baseline_id, control_family, analysis_role,
              added_feature_set, columns):
    columns = dedupe_preserve_order(columns)
    assert model_id not in model_features, f"Duplicate model_id: {model_id}"
    assert columns and not [c for c in columns if c not in df.columns]
    model_features[model_id] = columns
    model_records.append({
        "task": task,
        "model_id": model_id,
        "baseline_id": baseline_id,
        "control_family": control_family,
        "analysis_role": analysis_role,
        "added_feature_set": added_feature_set,
        "n_features_manifest": len(columns),
    })

ptal_base_id = "PTAL_spatial_baseline"
add_model(
    "PTAL", ptal_base_id, None, "PTAL_spatial",
    "control_baseline", None, feature_sets[ptal_base_id]
)
for rep in increment_rep_sets:
    add_model(
        "PTAL", f"{ptal_base_id}__plus__{rep}", ptal_base_id,
        "PTAL_spatial", representation_class[rep], rep,
        feature_sets[ptal_base_id] + feature_sets[rep],
    )

epc_sparse_id = "EPC_controls_sparse"
epc_extensive_id = "EPC_controls_extensive"
add_model(
    "EPC", epc_sparse_id, None, "EPC_sparse",
    "control_baseline", None, feature_sets[epc_sparse_id]
)
for rep in increment_rep_sets:
    add_model(
        "EPC", f"{epc_sparse_id}__plus__{rep}", epc_sparse_id,
        "EPC_sparse", representation_class[rep], rep,
        feature_sets[epc_sparse_id] + feature_sets[rep],
    )

# The extensive model is both an explanatory ablation target relative to sparse
# and the matching baseline for the extensive-controls representation models.
add_model(
    "EPC", epc_extensive_id, epc_sparse_id, "EPC_extensive",
    "control_ablation_and_extensive_baseline", "floor_area_plus_construction_age",
    feature_sets[epc_extensive_id],
)
for rep in increment_rep_sets:
    add_model(
        "EPC", f"{epc_extensive_id}__plus__{rep}", epc_extensive_id,
        "EPC_extensive", representation_class[rep], rep,
        feature_sets[epc_extensive_id] + feature_sets[rep],
    )

add_model(
    "EPC", "EPC_controls_sparse__plus__floor_area", epc_sparse_id,
    "EPC_control_ablation", "control_ablation", "floor_area",
    epc_sparse + ["median_total_floor_area"],
)
add_model(
    "EPC", "EPC_controls_sparse__plus__construction_age", epc_sparse_id,
    "EPC_control_ablation", "control_ablation", "construction_age",
    epc_sparse + ["construction_age_band_mode"],
)

model_specs = pd.DataFrame(model_records)
assert len(model_specs) == 41
assert model_specs["model_id"].is_unique
assert model_specs.groupby("task").size().to_dict() == {"EPC": 28, "PTAL": 13}
assert set(model_specs.loc[model_specs["baseline_id"].notna(), "baseline_id"]).issubset(
    set(model_specs["model_id"])
)

display(model_specs)
print("Frozen model specifications:", len(model_specs))

## 3. Treat Street View coverage consistently

Locations without Street View imagery retain missing visual vectors for training-based filling. Only the known structural metadata are set directly: zero contributing images and the maximum search distance when no image is available.

In [ ]:
# Apply the same Street View coverage rules used in the main benchmark.
SV_META_COLS = [
    "sv_has_streetview", "sv_n_images", "sv_min_dist_m", "sv_mean_dist_m"
]
SV_CLIP_COLS = feature_sets["StreetView_CLIP_only"]

def apply_structural_sv_metadata_fill(task_df, task):
    out = task_df.copy()
    assert all(c in out.columns for c in SV_META_COLS)
    radius = {
        "PTAL": float(STREETVIEW_PTAL_RADIUS_M),
        "EPC": float(STREETVIEW_EPC_RADIUS_M),
    }[task]

    has = pd.to_numeric(out["sv_has_streetview"], errors="coerce")
    clip_complete = out[SV_CLIP_COLS].notna().all(axis=1)
    clip_all_missing = out[SV_CLIP_COLS].isna().all(axis=1)
    has = has.where(has.notna(), clip_complete.astype(int)).astype(int)

    assert has.isin([0, 1]).all()
    assert clip_complete[has.eq(1)].all(), f"{task}: flagged imagery with incomplete CLIP"
    assert clip_all_missing[has.eq(0)].all(), f"{task}: no-image flag with partial CLIP"

    out["sv_has_streetview"] = has
    out["sv_n_images"] = pd.to_numeric(out["sv_n_images"], errors="coerce")
    out["sv_min_dist_m"] = pd.to_numeric(out["sv_min_dist_m"], errors="coerce")
    out["sv_mean_dist_m"] = pd.to_numeric(out["sv_mean_dist_m"], errors="coerce")

    no_sv = has.eq(0)
    out.loc[no_sv, "sv_n_images"] = 0.0
    for c in ["sv_min_dist_m", "sv_mean_dist_m"]:
        out.loc[no_sv & out[c].isna(), c] = radius

    assert out.loc[no_sv, "sv_n_images"].eq(0).all()
    assert out.loc[no_sv, ["sv_min_dist_m", "sv_mean_dist_m"]].notna().all().all()
    assert (out.loc[has.eq(1), "sv_n_images"].dropna() >= 1).all()
    return out

sv_audit_rows = []
for task in ["PTAL", "EPC"]:
    task_filled = apply_structural_sv_metadata_fill(df[df["task"] == task], task)
    has = task_filled["sv_has_streetview"].eq(1)
    sv_audit_rows.append({
        "task": task,
        "n": int(len(task_filled)),
        "n_with_streetview": int(has.sum()),
        "coverage_pct": float(100 * has.mean()),
        "n_no_streetview": int((~has).sum()),
        "covered_rows_with_missing_metadata": int(
            task_filled.loc[has, SV_META_COLS[1:]].isna().any(axis=1).sum()
        ),
    })
sv_audit = pd.DataFrame(sv_audit_rows)
display(sv_audit)

## 4. Use the same training procedure as the main benchmark

Missing-value treatment, scaling, categorical encoding and Ridge penalty selection are performed within the training data for each borough round. No information from the held-out boroughs is used to prepare the predictors.

In [ ]:
# Build the training-only preprocessing and Ridge pipeline.
def make_onehot():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)

def build_pipeline(feature_cols):
    categorical_cols = [c for c in feature_cols if c in categorical_master]
    numeric_cols = [c for c in feature_cols if c not in categorical_master]
    transformers = []

    if numeric_cols:
        transformers.append((
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]),
            numeric_cols,
        ))
    if categorical_cols:
        transformers.append((
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="constant", fill_value="__MISSING__")),
                ("onehot", make_onehot()),
            ]),
            categorical_cols,
        ))

    pre = ColumnTransformer(
        transformers=transformers,
        remainder="drop",
        sparse_threshold=0.0,
    )
    return Pipeline([
        ("preprocess", pre),
        ("ridge", Ridge(solver="lsqr", max_iter=5000, tol=1e-4)),
    ])

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def atomic_csv(frame, path):
    tmp = path.with_name(path.stem + ".tmp" + path.suffix)
    frame.to_csv(tmp, index=False)
    tmp.replace(path)

def atomic_parquet(frame, path):
    tmp = path.with_name(path.stem + ".tmp" + path.suffix)
    frame.to_parquet(tmp, index=False)
    tmp.replace(path)

def atomic_json(obj, path):
    tmp = path.with_name(path.stem + ".tmp" + path.suffix)
    with open(tmp, "w") as f:
        json.dump(obj, f, indent=2)
    tmp.replace(path)

def coerce_bool(series):
    if pd.api.types.is_bool_dtype(series):
        return series
    mapped = series.astype(str).str.strip().str.lower().map({"true": True, "false": False})
    assert mapped.notna().all(), "Unexpected boolean encoding."
    return mapped.astype(bool)

## 5. Reuse the same held-out boroughs

The five borough-based divisions are reconstructed and compared row by row with the main benchmark assignments. This makes every baseline-versus-added-representation difference a comparison on exactly the same locations.

In [ ]:
# Reconstruct the same held-out boroughs and record the model specification.
outer_assignment_rows = []
fold_index_by_task = {}

for task in ["PTAL", "EPC"]:
    task_df = df[df["task"] == task].reset_index(drop=True)
    groups = task_df[group_col].astype(str).to_numpy()
    splitter = GroupKFold(n_splits=RIDGE_OUTER_SPLITS)
    task_fold = np.full(len(task_df), -1, dtype=int)

    for fold, (_, test_idx) in enumerate(splitter.split(task_df, task_df[target_col], groups)):
        task_fold[test_idx] = fold
        for idx in test_idx:
            outer_assignment_rows.append({
                "sample_id": task_df.loc[idx, "sample_id"],
                "task": task,
                "outer_fold": int(fold),
                "borough_code": task_df.loc[idx, group_col],
                "target": float(task_df.loc[idx, target_col]),
            })
    assert (task_fold >= 0).all()
    fold_index_by_task[task] = task_fold

outer_folds = (
    pd.DataFrame(outer_assignment_rows)
    .sort_values(["task", "sample_id"], kind="mergesort")
    .reset_index(drop=True)
)
saved_outer_folds = (
    pd.read_csv(RIDGE_OUTER_FOLDS_PATH)
    .sort_values(["task", "sample_id"], kind="mergesort")
    .reset_index(drop=True)
)
pd.testing.assert_frame_equal(
    saved_outer_folds[outer_folds.columns],
    outer_folds,
    check_dtype=False,
    check_exact=False,
    rtol=0,
    atol=1e-12,
)
assert outer_folds.groupby(["task", "borough_code"])["outer_fold"].nunique().eq(1).all()

fold_assignment_hash = hashlib.sha256(
    pd.util.hash_pandas_object(outer_folds, index=False).values.tobytes()
).hexdigest()
assert source_06_spec["outer_fold_assignment_sha256"] == fold_assignment_hash
assert source_06_audit["outer_fold_assignment_sha256"] == fold_assignment_hash

def columns_sha256(columns):
    return hashlib.sha256(
        json.dumps(list(columns), separators=(",", ":")).encode("utf-8")
    ).hexdigest()

model_spec_payload = []
for row in model_specs.itertuples(index=False):
    model_spec_payload.append({
        "task": row.task,
        "model_id": row.model_id,
        "baseline_id": row.baseline_id,
        "control_family": row.control_family,
        "analysis_role": row.analysis_role,
        "added_feature_set": row.added_feature_set,
        "n_features_manifest": int(row.n_features_manifest),
        "feature_columns_sha256": columns_sha256(model_features[row.model_id]),
    })

dinov3_decision_rule = {
    "status_before_07": "deferred",
    "trigger": (
        "Run a limited DINOv3 backbone/crop-scale sensitivity only if a pre-specified "
        "DINOv2 controls-plus comparison achieves mean paired delta R2 >= 0.01 and "
        "R2 improvement in at least 4 of 5 frozen outer folds."
    ),
    "comparisons": [
        "PTAL_spatial_baseline__plus__DINOv2",
        "EPC_controls_sparse__plus__DINOv2",
        "EPC_controls_extensive__plus__DINOv2",
    ],
    "mean_delta_r2_threshold": 0.01,
    "minimum_r2_wins_out_of_5": 4,
}

run_spec = {
    "run_spec_version": "07-v1-2026-08-23",
    "notebook": "07_controls_incremental_value_and_control_ablation.ipynb",
    "source_notebook_06_run_spec": str(RIDGE_RUN_SPEC_PATH),
    "source_notebook_06_audit": str(RIDGE_CORE_AUDIT_PATH),
    "model_key_sha256": model_key_hash,
    "feature_manifest_sha256": manifest_hash,
    "outer_fold_assignment_sha256": fold_assignment_hash,
    "sklearn_version": sklearn.__version__,
    "target_column": target_col,
    "group_column": group_col,
    "outer_splits": int(RIDGE_OUTER_SPLITS),
    "inner_splits": int(RIDGE_INNER_SPLITS),
    "alpha_grid": [float(x) for x in RIDGE_ALPHA_GRID],
    "inner_selection_metric": "RMSE",
    "primary_increment_metrics": ["paired_delta_r2", "paired_delta_rmse", "paired_delta_mae"],
    "fold_inference": "descriptive mean, SD and win counts; no independent-fold p-values",
    "model_specifications": model_spec_payload,
    "epc_extensive_baseline_label": "privileged within-EPC tabular baseline",
    "dinov3_decision_rule": dinov3_decision_rule,
    "preprocessing": {
        "numeric_imputation": "training-fold median",
        "numeric_scaling": "training-fold StandardScaler",
        "categorical_imputation": "training-fold constant __MISSING__",
        "categorical_encoding": "training-fold OneHotEncoder(handle_unknown=ignore)",
        "streetview_clip_missing": "training-fold median",
    },
}
run_spec_sha256 = hashlib.sha256(
    json.dumps(run_spec, sort_keys=True).encode("utf-8")
).hexdigest()

if INCREMENTAL_RUN_SPEC_PATH.exists():
    with open(INCREMENTAL_RUN_SPEC_PATH, "r") as f:
        existing_spec = json.load(f)
    assert existing_spec == run_spec, (
        "Existing Notebook-07 checkpoints belong to a different frozen run specification. "
        "Do not mix outputs; archive the old 07 outputs before a documented rerun."
    )
else:
    atomic_json(run_spec, INCREMENTAL_RUN_SPEC_PATH)

expected_runs_df = pd.DataFrame([
    {"task": row.task, "model_id": row.model_id, "outer_fold": fold}
    for row in model_specs.itertuples(index=False)
    for fold in range(RIDGE_OUTER_SPLITS)
])
expected_keys = set(map(
    tuple, expected_runs_df[["task", "model_id", "outer_fold"]].to_numpy()
))
expected_prediction_rows = int(sum(
    int(task_counts[task]) * int((model_specs["task"] == task).sum())
    for task in ["PTAL", "EPC"]
))
expected_paired_rows = int(
    model_specs["baseline_id"].notna().sum() * RIDGE_OUTER_SPLITS
)

assert len(expected_keys) == 205
assert expected_prediction_rows == 645761
assert expected_paired_rows == 195

print("Verified exact Notebook-06 fold SHA256:", fold_assignment_hash)
print("Notebook-07 run-spec SHA256:", run_spec_sha256)
print("Expected outer-fold runs:", len(expected_keys))
print("Expected prediction rows:", expected_prediction_rows)
print("Expected paired-delta rows:", expected_paired_rows)

## 6. Fit the incremental models

For each task, baseline and added representation, the model is fitted in all five borough rounds. Per-round predictions and metrics are saved as the analysis runs, with stored model and sample identities preventing incompatible partial results from being combined.

In [ ]:
# Fit each incremental model and save round-level results and predictions.
if INCREMENTAL_RESULTS_PATH.exists():
    completed = pd.read_csv(INCREMENTAL_RESULTS_PATH)
    required_result_cols = {"task", "model_id", "outer_fold", "run_spec_sha256"}
    assert required_result_cols.issubset(completed.columns)
    assert not completed.duplicated(["task", "model_id", "outer_fold"]).any()
    assert completed["run_spec_sha256"].eq(run_spec_sha256).all()
    completed["outer_fold"] = completed["outer_fold"].astype(int)
    completed_keys = set(map(
        tuple, completed[["task", "model_id", "outer_fold"]].to_numpy()
    ))
    assert completed_keys.issubset(expected_keys)
else:
    completed = pd.DataFrame()

result_rows = [] if completed.empty else completed.to_dict("records")

def checkpoint_is_valid(path, task, model_id, outer_fold, expected_ids):
    if not path.exists():
        return False
    try:
        p = pd.read_parquet(path)
        required = {
            "sample_id", "task", "model_id", "outer_fold", "borough_code",
            "y_true", "y_pred", "run_spec_sha256",
        }
        if not required.issubset(p.columns) or p["sample_id"].duplicated().any():
            return False
        if not p["task"].eq(task).all() or not p["model_id"].eq(model_id).all():
            return False
        if not p["outer_fold"].astype(int).eq(outer_fold).all():
            return False
        if not p["run_spec_sha256"].eq(run_spec_sha256).all():
            return False
        if not np.isfinite(p["y_true"]).all() or not np.isfinite(p["y_pred"]).all():
            return False
        return set(p["sample_id"].astype(str)) == set(pd.Series(expected_ids).astype(str))
    except Exception:
        return False

for task in ["PTAL", "EPC"]:
    task_df = df[df["task"] == task].reset_index(drop=True)
    task_df = apply_structural_sv_metadata_fill(task_df, task)
    y = pd.to_numeric(task_df[target_col], errors="raise").to_numpy()
    groups = task_df[group_col].astype(str).to_numpy()
    task_fold = fold_index_by_task[task]

    task_specs = model_specs[model_specs["task"] == task]
    for spec_row in task_specs.itertuples(index=False):
        model_id = spec_row.model_id
        cols = model_features[model_id]
        X = task_df[cols]

        for outer_fold in range(RIDGE_OUTER_SPLITS):
            test_idx = np.flatnonzero(task_fold == outer_fold)
            train_idx = np.flatnonzero(task_fold != outer_fold)
            run_key = (task, model_id, outer_fold)
            pred_file = INCREMENTAL_CHUNK_DIR / f"{task}__{model_id}__fold{outer_fold}.parquet"

            completed_keys_now = {
                (r["task"], r["model_id"], int(r["outer_fold"]))
                for r in result_rows
            }
            checkpoint_ok = checkpoint_is_valid(
                pred_file, task, model_id, outer_fold,
                task_df.iloc[test_idx]["sample_id"].to_numpy(),
            )
            if run_key in completed_keys_now and checkpoint_ok:
                print("SKIP validated checkpoint:", run_key)
                continue

            print("\n" + "=" * 100)
            print(task, "|", model_id, "| outer fold", outer_fold)
            print("=" * 100)

            X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
            y_train, y_test = y[train_idx], y[test_idx]
            g_train = groups[train_idx]
            assert len(np.unique(g_train)) >= RIDGE_INNER_SPLITS

            inner = GroupKFold(n_splits=RIDGE_INNER_SPLITS)
            inner_splits = list(inner.split(X_train, y_train, g_train))
            for inner_train, inner_valid in inner_splits:
                assert set(g_train[inner_train]).isdisjoint(set(g_train[inner_valid]))

            pipe = build_pipeline(cols)
            search = GridSearchCV(
                estimator=pipe,
                param_grid={"ridge__alpha": RIDGE_ALPHA_GRID},
                scoring="neg_root_mean_squared_error",
                cv=inner_splits,
                refit=True,
                n_jobs=1,
                return_train_score=False,
                error_score="raise",
            )

            t0 = time.time()
            search.fit(X_train, y_train)
            elapsed_s = time.time() - t0
            pred = search.predict(X_test)
            assert len(pred) == len(test_idx)
            assert np.isfinite(pred).all()

            best_alpha = float(search.best_params_["ridge__alpha"])
            alpha_edge = best_alpha in {
                float(min(RIDGE_ALPHA_GRID)), float(max(RIDGE_ALPHA_GRID))
            }
            row = {
                "task": task,
                "model_id": model_id,
                "baseline_id": spec_row.baseline_id,
                "control_family": spec_row.control_family,
                "analysis_role": spec_row.analysis_role,
                "added_feature_set": spec_row.added_feature_set,
                "outer_fold": int(outer_fold),
                "n_features_manifest": int(len(cols)),
                "n_train": int(len(train_idx)),
                "n_test": int(len(test_idx)),
                "n_train_boroughs": int(len(np.unique(g_train))),
                "n_test_boroughs": int(len(np.unique(groups[test_idx]))),
                "best_alpha": best_alpha,
                "alpha_grid_edge": bool(alpha_edge),
                "inner_best_rmse": float(-search.best_score_),
                "r2": float(r2_score(y_test, pred)),
                "rmse": rmse(y_test, pred),
                "mae": float(mean_absolute_error(y_test, pred)),
                "fit_seconds": float(elapsed_s),
                "run_spec_sha256": run_spec_sha256,
            }

            pred_frame = pd.DataFrame({
                "sample_id": task_df.iloc[test_idx]["sample_id"].to_numpy(),
                "task": task,
                "model_id": model_id,
                "outer_fold": outer_fold,
                "borough_code": groups[test_idx],
                "y_true": y_test,
                "y_pred": pred,
                "run_spec_sha256": run_spec_sha256,
            })
            atomic_parquet(pred_frame, pred_file)

            result_rows = [
                r for r in result_rows
                if (r["task"], r["model_id"], int(r["outer_fold"])) != run_key
            ]
            result_rows.append(row)
            results_now = pd.DataFrame(result_rows).sort_values(
                ["task", "model_id", "outer_fold"], kind="mergesort"
            )
            atomic_csv(results_now, INCREMENTAL_RESULTS_PATH)
            print(row)

            del search, pipe, X_train, X_test, pred, pred_frame
            gc.collect()

print("Notebook-07 nested Ridge analysis is complete or safely checkpointed.")

## 7. Assemble complete prediction sets

The analysis expects 205 model rounds and 645,761 held-out predictions. Each saved prediction set is matched to the intended model, task, borough round, outcome and sample IDs before the combined summaries are created.

In [ ]:
# Verify complete and consistent predictions before calculating comparisons.
results = pd.read_csv(INCREMENTAL_RESULTS_PATH)
results["outer_fold"] = results["outer_fold"].astype(int)
assert not results.duplicated(["task", "model_id", "outer_fold"]).any()
assert results["run_spec_sha256"].eq(run_spec_sha256).all()
actual_keys = set(map(
    tuple, results[["task", "model_id", "outer_fold"]].to_numpy()
))

missing_run_keys = sorted(expected_keys - actual_keys)
unexpected_run_keys = sorted(actual_keys - expected_keys)
print("Completed fold-runs:", len(actual_keys), "/", len(expected_keys))
print("Missing run keys:", len(missing_run_keys))
print("Unexpected run keys:", len(unexpected_run_keys))
assert not missing_run_keys, "Notebook 07 incomplete: rerun Section 6."
assert not unexpected_run_keys

pred_frames = []
chunk_audit_rows = []
for task, model_id, outer_fold in sorted(expected_keys):
    path = INCREMENTAL_CHUNK_DIR / f"{task}__{model_id}__fold{outer_fold}.parquet"
    assert path.exists(), f"Missing prediction chunk: {path.name}"
    p = pd.read_parquet(path)
    required = {
        "sample_id", "task", "model_id", "outer_fold", "borough_code",
        "y_true", "y_pred", "run_spec_sha256",
    }
    assert required.issubset(p.columns), f"Malformed chunk: {path.name}"
    assert p["task"].eq(task).all()
    assert p["model_id"].eq(model_id).all()
    assert p["outer_fold"].astype(int).eq(outer_fold).all()
    assert p["run_spec_sha256"].eq(run_spec_sha256).all()
    assert p["sample_id"].is_unique
    assert np.isfinite(p["y_true"]).all() and np.isfinite(p["y_pred"]).all()

    expected_fold = outer_folds[
        (outer_folds["task"] == task)
        & (outer_folds["outer_fold"] == outer_fold)
    ][["sample_id", "borough_code", "target"]].sort_values(
        "sample_id"
    ).reset_index(drop=True)
    observed_fold = p[
        ["sample_id", "borough_code", "y_true"]
    ].sort_values("sample_id").reset_index(drop=True)

    assert expected_fold["sample_id"].equals(observed_fold["sample_id"])
    assert expected_fold["borough_code"].astype(str).equals(
        observed_fold["borough_code"].astype(str)
    )
    assert np.allclose(
        expected_fold["target"], observed_fold["y_true"], rtol=0, atol=1e-12
    )

    chunk_audit_rows.append({
        "task": task,
        "model_id": model_id,
        "outer_fold": int(outer_fold),
        "n_rows": int(len(p)),
        "file": path.name,
    })
    pred_frames.append(p)

preds = pd.concat(pred_frames, ignore_index=True)
assert not preds.duplicated(
    ["sample_id", "task", "model_id", "outer_fold"]
).any()
assert len(preds) == expected_prediction_rows

print("Validated prediction chunks:", len(chunk_audit_rows))
print("Validated prediction rows:", len(preds))

## 8. Calculate added value and the EPC control contributions

For R², a positive difference favours the model with the added information. For RMSE and MAE, a negative difference indicates lower error. The summary reports the average within-round change, its variability and the number of rounds in which the augmented model improves.

The EPC component comparison adds floor area, construction age and both together to the same compact baseline. Its purpose is to explain the strong richer baseline, not to redefine the representation benchmark.

In [ ]:
# Summarise absolute performance, within-round gains and EPC control contributions.
summary_rows = []
for (task, model_id), g in results.groupby(["task", "model_id"]):
    pred_g = preds[(preds["task"] == task) & (preds["model_id"] == model_id)]
    spec = model_specs.loc[model_specs["model_id"] == model_id].iloc[0]
    assert len(g) == RIDGE_OUTER_SPLITS
    assert len(pred_g) == task_counts[task]

    summary_rows.append({
        "task": task,
        "model_id": model_id,
        "baseline_id": spec["baseline_id"],
        "control_family": spec["control_family"],
        "analysis_role": spec["analysis_role"],
        "added_feature_set": spec["added_feature_set"],
        "n_features": int(spec["n_features_manifest"]),
        "mean_r2": float(g["r2"].mean()),
        "sd_r2": float(g["r2"].std(ddof=1)),
        "mean_rmse": float(g["rmse"].mean()),
        "sd_rmse": float(g["rmse"].std(ddof=1)),
        "mean_mae": float(g["mae"].mean()),
        "sd_mae": float(g["mae"].std(ddof=1)),
        "pooled_r2": float(r2_score(pred_g["y_true"], pred_g["y_pred"])),
        "pooled_rmse": rmse(pred_g["y_true"], pred_g["y_pred"]),
        "pooled_mae": float(mean_absolute_error(pred_g["y_true"], pred_g["y_pred"])),
        "alpha_edge_hits": int(coerce_bool(g["alpha_grid_edge"]).sum()),
        "median_best_alpha": float(g["best_alpha"].median()),
        "total_fit_minutes": float(g["fit_seconds"].sum() / 60),
    })

summary = pd.DataFrame(summary_rows).sort_values(
    ["task", "control_family", "mean_r2"],
    ascending=[True, True, False],
    kind="mergesort",
)

paired_rows = []
for spec in model_specs[model_specs["baseline_id"].notna()].itertuples(index=False):
    model_g = results[
        (results["task"] == spec.task) & (results["model_id"] == spec.model_id)
    ][["outer_fold", "r2", "rmse", "mae"]].copy()
    base_g = results[
        (results["task"] == spec.task) & (results["model_id"] == spec.baseline_id)
    ][["outer_fold", "r2", "rmse", "mae"]].copy()
    paired = model_g.merge(
        base_g, on="outer_fold", how="inner", validate="one_to_one",
        suffixes=("_model", "_baseline"),
    )
    assert len(paired) == RIDGE_OUTER_SPLITS

    for row in paired.itertuples(index=False):
        delta_r2 = float(row.r2_model - row.r2_baseline)
        delta_rmse = float(row.rmse_model - row.rmse_baseline)
        delta_mae = float(row.mae_model - row.mae_baseline)
        paired_rows.append({
            "task": spec.task,
            "model_id": spec.model_id,
            "baseline_id": spec.baseline_id,
            "control_family": spec.control_family,
            "analysis_role": spec.analysis_role,
            "added_feature_set": spec.added_feature_set,
            "outer_fold": int(row.outer_fold),
            "model_r2": float(row.r2_model),
            "baseline_r2": float(row.r2_baseline),
            "delta_r2": delta_r2,
            "model_rmse": float(row.rmse_model),
            "baseline_rmse": float(row.rmse_baseline),
            "delta_rmse": delta_rmse,
            "model_mae": float(row.mae_model),
            "baseline_mae": float(row.mae_baseline),
            "delta_mae": delta_mae,
            "r2_win": bool(delta_r2 > 0),
            "rmse_win": bool(delta_rmse < 0),
            "mae_win": bool(delta_mae < 0),
        })

paired_deltas = pd.DataFrame(paired_rows).sort_values(
    ["task", "model_id", "outer_fold"], kind="mergesort"
)
assert len(paired_deltas) == expected_paired_rows

absolute_lookup = summary.set_index("model_id")
delta_summary_rows = []
for (task, model_id, baseline_id), g in paired_deltas.groupby(
    ["task", "model_id", "baseline_id"]
):
    spec = model_specs.loc[model_specs["model_id"] == model_id].iloc[0]
    delta_summary_rows.append({
        "task": task,
        "model_id": model_id,
        "baseline_id": baseline_id,
        "control_family": spec["control_family"],
        "analysis_role": spec["analysis_role"],
        "added_feature_set": spec["added_feature_set"],
        "mean_delta_r2": float(g["delta_r2"].mean()),
        "sd_delta_r2": float(g["delta_r2"].std(ddof=1)),
        "r2_wins_out_of_5": int(g["r2_win"].sum()),
        "mean_delta_rmse": float(g["delta_rmse"].mean()),
        "sd_delta_rmse": float(g["delta_rmse"].std(ddof=1)),
        "rmse_wins_out_of_5": int(g["rmse_win"].sum()),
        "mean_delta_mae": float(g["delta_mae"].mean()),
        "sd_delta_mae": float(g["delta_mae"].std(ddof=1)),
        "mae_wins_out_of_5": int(g["mae_win"].sum()),
        "pooled_delta_r2": float(
            absolute_lookup.loc[model_id, "pooled_r2"]
            - absolute_lookup.loc[baseline_id, "pooled_r2"]
        ),
        "pooled_delta_rmse": float(
            absolute_lookup.loc[model_id, "pooled_rmse"]
            - absolute_lookup.loc[baseline_id, "pooled_rmse"]
        ),
        "pooled_delta_mae": float(
            absolute_lookup.loc[model_id, "pooled_mae"]
            - absolute_lookup.loc[baseline_id, "pooled_mae"]
        ),
    })

delta_summary = pd.DataFrame(delta_summary_rows).sort_values(
    ["task", "control_family", "mean_delta_r2"],
    ascending=[True, True, False],
    kind="mergesort",
)
assert len(delta_summary) == int(model_specs["baseline_id"].notna().sum()) == 39

ablation_ids = [
    "EPC_controls_sparse__plus__floor_area",
    "EPC_controls_sparse__plus__construction_age",
    "EPC_controls_extensive",
]
epc_control_ablation = delta_summary[
    delta_summary["model_id"].isin(ablation_ids)
].copy()
assert len(epc_control_ablation) == 3

atomic_csv(summary, INCREMENTAL_SUMMARY_PATH)
atomic_csv(paired_deltas, INCREMENTAL_DELTAS_PATH)
atomic_csv(delta_summary, INCREMENTAL_DELTA_SUMMARY_PATH)
atomic_csv(epc_control_ablation, INCREMENTAL_ABLATION_SUMMARY_PATH)
atomic_parquet(preds, INCREMENTAL_PREDICTIONS_PATH)

print("Absolute performance by task/control family")
display(summary)
print("Fold-paired incremental value")
display(delta_summary)
print("Explanatory EPC control ablation")
display(epc_control_ablation)

## 9. Finalise the results

All planned model rounds and comparisons must be present, and the Ridge penalty range must be adequate for interpretation. The completed outputs contain all 205 runs and show no repeated penalty-boundary problem.

In [ ]:
# Validate the run matrix and penalty range, then save the final summaries.
counts = results.groupby(["task", "model_id"]).size().rename("n_outer_folds").reset_index()
edge_counts = (
    results.assign(alpha_grid_edge=coerce_bool(results["alpha_grid_edge"]))
    .groupby(["task", "model_id"])["alpha_grid_edge"]
    .sum().rename("edge_hits").reset_index()
)
repeated_edge = edge_counts[edge_counts["edge_hits"] >= 3].copy()

dinov2_trigger_rows = delta_summary[
    delta_summary["model_id"].isin(dinov3_decision_rule["comparisons"])
].copy()
assert len(dinov2_trigger_rows) == 3
dinov2_trigger_rows["dinov3_trigger_pass"] = (
    dinov2_trigger_rows["mean_delta_r2"].ge(
        dinov3_decision_rule["mean_delta_r2_threshold"]
    )
    & dinov2_trigger_rows["r2_wins_out_of_5"].ge(
        dinov3_decision_rule["minimum_r2_wins_out_of_5"]
    )
)
dinov3_recommended = bool(dinov2_trigger_rows["dinov3_trigger_pass"].any())

audit = {
    "run_spec_path": str(INCREMENTAL_RUN_SPEC_PATH),
    "run_spec_sha256": run_spec_sha256,
    "model_key_sha256": model_key_hash,
    "feature_manifest_sha256": manifest_hash,
    "outer_fold_assignment_sha256": fold_assignment_hash,
    "source_06_integrity_gate_pass": bool(source_06_audit["integrity_gate_pass"]),
    "source_06_interpretation_gate_pass": bool(source_06_audit["interpretation_gate_pass"]),
    "expected_fold_runs": int(len(expected_keys)),
    "completed_fold_runs": int(len(actual_keys)),
    "all_models_have_expected_outer_folds": bool(
        counts["n_outer_folds"].eq(RIDGE_OUTER_SPLITS).all()
        and len(counts) == len(model_specs)
    ),
    "expected_prediction_rows": int(expected_prediction_rows),
    "actual_prediction_rows": int(len(preds)),
    "prediction_rows_match_expected": bool(len(preds) == expected_prediction_rows),
    "prediction_chunks_validated": int(len(chunk_audit_rows)),
    "expected_paired_delta_rows": int(expected_paired_rows),
    "actual_paired_delta_rows": int(len(paired_deltas)),
    "paired_deltas_complete": bool(len(paired_deltas) == expected_paired_rows),
    "paired_delta_models": int(len(delta_summary)),
    "epc_control_ablation_models": int(len(epc_control_ablation)),
    "n_alpha_grid_edge_hits": int(edge_counts["edge_hits"].sum()),
    "n_models_with_repeated_edge_hits": int(len(repeated_edge)),
    "alpha_grid_adequacy_pass": bool(repeated_edge.empty),
    "dinov3_decision_rule": dinov3_decision_rule,
    "dinov3_trigger_evaluations": dinov2_trigger_rows[
        ["model_id", "mean_delta_r2", "r2_wins_out_of_5", "dinov3_trigger_pass"]
    ].to_dict("records"),
    "dinov3_recommended_by_prespecified_trigger": dinov3_recommended,
    "epc_extensive_baseline_label": "privileged within-EPC tabular baseline",
    "primary_validation_description": "same frozen borough-grouped nested spatial CV as Notebook 06",
    "incremental_inference": "fold-paired descriptive deltas and wins; no independent-fold p-values",
    "integrity_gate_pass": True,
    "interpretation_gate_pass": bool(repeated_edge.empty),
}
atomic_json(audit, INCREMENTAL_AUDIT_PATH)

display(pd.Series(audit, name="value"))
display(dinov2_trigger_rows)

if not repeated_edge.empty:
    display(repeated_edge)
    raise RuntimeError(
        "Integrity PASS, but interpretation is on hold: at least one model selected "
        "an alpha-grid boundary in >=3 outer folds. Expand only the alpha grid in a "
        "documented rerun; do not change the model matrix or folds."
    )

print("07 incremental-value analysis — integrity gate: PASS")
print("07 incremental-value analysis — alpha adequacy / interpretation gate: PASS")
print("DINOv3 recommended by frozen practical trigger:", dinov3_recommended)

## Interpretation

Representations provide clear additional information for PTAL and for EPC when only compact property controls are available. Their contribution is much smaller once EPC construction age is known directly. This distinction is important for the dissertation: the results do not imply that representations are universally weak for EPC; rather, their value depends on which property records are already available. The richer EPC model should therefore be presented as an information-rich reference, while the compact-control comparison better describes the additional value of remotely observed features.